In [ ]:
%pip install mteb==2.10.12 accelerate>=1.1.0 sentence-transformers==5.3.0 dotenv torch==2.10.0+cu128 numpy==2.0.2 datasets==4.8.3 requests==2.32.4 scikit-learn==1.6.1 scipy==1.16.3 polars==1.35.2

In [1]:
import mteb
import pandas as pd
import os
from sentence_transformers import SentenceTransformer
import torch
from dotenv import load_dotenv

load_dotenv()

/Users/lucasokamura/miniconda3/envs/probing-env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

In [2]:
DEVICE = "cuda" if torch.cuda.is_available() else "mps" if torch.mps.is_available() else "cpu"
DEVICE

'mps'

In [3]:
tasks = mteb.get_tasks(languages=["por"], modalities=['text'])

for task in tasks:
    print(task)

BibleNLPBitextMining(name='BibleNLPBitextMining', languages=['eng', 'por'])
FloresBitextMining(name='FloresBitextMining', languages=['ace', 'acm', 'acq', '...'])
NTREXBitextMining(name='NTREXBitextMining', languages=['arb', 'ben', 'cat', '...'])
TatoebaBitextMining(name='Tatoeba', languages=['eng', 'por'])
WebFAQBitextMiningQAs(name='WebFAQBitextMiningQAs', languages=['cat', 'dan', 'deu', '...'])
WebFAQBitextMiningQuestions(name='WebFAQBitextMiningQuestions', languages=['cat', 'dan', 'deu', '...'])
AfriSentiClassification(name='AfriSentiClassification', languages=['por'])
AfriSentiLangClassification(name='AfriSentiLangClassification', languages=['amh', 'arq', 'ary', '...'])
LanguageClassification(name='LanguageClassification', languages=['ara', 'bul', 'cmn', '...'])
MassiveIntentClassification(name='MassiveIntentClassification', languages=['por'])
MassiveScenarioClassification(name='MassiveScenarioClassification', languages=['por'])
MultiHateClassification(name='MultiHateClassification

In [4]:
model_list = ["iara-project/NeoBERTugues-matryoshka-sts-pt"]
dims_list = [64]
# None, 512, 256, 128, 
filepath = '../data/results_eval_mteb_finetuned.csv'

tasks = [
    ("Assin2STS", None),
    ("SICK-BR-STS", None),
    ("STSBenchmarkMultilingualSTS", 'pt'),
    
    ('MassiveIntentClassification', 'pt'),
    ('MultiHateClassification', 'por'),
    ('BrazilianToxicTweetsClassification', None),
    ('HateSpeechPortugueseClassification', None),
    ('TweetSentimentClassification', 'portuguese'),

    ('MultiLongDocReranking', 'pt'),
    ('WikipediaRerankingMultilingual', 'pt'),
    ('XGlueWPRReranking', 'pt'),

    ('WebFAQRetrieval', 'por'),
    ('MultiLongDocRetrieval', 'pt'),
    ('WikipediaRetrievalMultilingual', 'pt')
]

In [12]:
model_meta = SentenceTransformer("rufimelo/Legal-BERTimbau-sts-large-ma-v3", device=DEVICE)

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 9180.12it/s]
BertModel LOAD REPORT from: rufimelo/Legal-BERTimbau-sts-large-ma-v3
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Invalid model-index. Not loading eval results into CardData.


In [17]:
model_meta.encode("oi", truncate_dim=10).shape

(10,)

In [ ]:
for model_name in model_list:
    model_meta = SentenceTransformer(model_name, device=DEVICE)
    for truncate_dim in dims_list:
        # select the desired tasks and evaluate
        task_name_list = []
        model_name_list = []
        main_score_list = []
        truncate_dims_list = []
        for task_info in tasks:
            print(f"""
#############################

[{model_name} - {truncate_dim} dims] Avaliando {task_info[0]} ({task_info[1]})...

#############################
            """)

            task = mteb.get_task(task_info[0], languages=['por'], hf_subsets=task_info[1])

            # with encode kwargs
            result = mteb.evaluate(model_meta, task, encode_kwargs={"batch_size": 8, "truncate_dim": truncate_dim}, cache=None)

            task_name = result.task_results[0].task_name
            model_name = result.model_name
            main_score = result.task_results[0].main_score

            task_name_list.append(task_name)
            model_name_list.append(model_name)
            main_score_list.append(main_score)
            truncate_dims_list.append(truncate_dim)

            print(f"Main Score: {main_score}")

            del task, result
            torch.cuda.empty_cache()
        
        df_results = pd.DataFrame({
            'model_name': model_name_list,
            'embedding_dim': truncate_dims_list,
            'task_name': task_name_list,
            'main_score': main_score_list
        })

        if os.path.exists(filepath):
            df_results_cache = pd.read_csv(filepath)
            df_results = pd.concat([df_results_cache, df_results], axis=0, ignore_index=True)

        df_results.to_csv(filepath, index=False)

        print(f"Avaliação concluída para {model_name} - {truncate_dim} dims!")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


Loading weights:   0%|          | 0/134 [00:00<?, ?it/s]


#############################

[iara-project/NeoBERTugues-matryoshka-sts-pt - 64 dims] Avaliando Assin2STS (None)...

#############################
            
Main Score: 0.777916

#############################

[iara-project/NeoBERTugues-matryoshka-sts-pt - 64 dims] Avaliando SICK-BR-STS (None)...

#############################
            
Main Score: 0.870959

#############################

[iara-project/NeoBERTugues-matryoshka-sts-pt - 64 dims] Avaliando STSBenchmarkMultilingualSTS (pt)...

#############################
            
Main Score: 0.790484

#############################

[iara-project/NeoBERTugues-matryoshka-sts-pt - 64 dims] Avaliando MassiveIntentClassification (pt)...

#############################
            
Main Score: 0.4541945

#############################

[iara-project/NeoBERTugues-matryoshka-sts-pt - 64 dims] Avaliando MultiHateClassification (por)...

#############################
            
Main Score: 0.5781

#############################

[iara-p

README.md: 0.00B [00:00, ?B/s]

pt-qrels/test-00000-of-00001.parquet:   0%|          | 0.00/102k [00:00<?, ?B/s]

...

Filtering queries by qrels:   0%|          | 0/200 [00:00<?, ? examples/s]

Filtering queries by qrels:   0%|          | 0/200 [00:00<?, ? examples/s]

Processing queries for dataloading:   0%|          | 0/200 [00:00<?, ? examples/s]

Converting corpus dict:   0%|          | 0/6569 [00:00<?, ? examples/s]

: 